In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import deadwood
import sklearn as sk
import scipy
import itertools

import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [24]:
methods=['one-class-svm','isolation-forest','dbscan','lof']

# Loader

In [20]:
import os
def load_all_datasets(directory):
    all_datasets = {}
    for root, dirs, files in os.walk(directory):
        for file in files:
            mat_data=scipy.io.loadmat(os.path.join(root,file))
            file_key = os.path.splitext(file)[0]
            df_x=pd.DataFrame(mat_data["X"])
            df_y=pd.DataFrame(mat_data["y"])

            duplicate_count = df_x.duplicated().sum()
            if duplicate_count > 0:
                print(f"Dataset '{file_key}' has {duplicate_count} duplicate rows (out of {df_x.shape[0]}. Removing them...")
                df_x_clean = df_x.drop_duplicates()
                df_y_clean = df_y.loc[df_x_clean.index]
                all_datasets[file_key]={
                    "X":df_x_clean.reset_index(drop=True),
                    "y":df_y_clean.reset_index(drop=True)
                }
            else:
                all_datasets[file_key]={
                    "X":pd.DataFrame(mat_data["X"]),
                    "y":pd.DataFrame(mat_data["y"])
                }
    return all_datasets


In [22]:
all_datasets = load_all_datasets("C:\\Users\\andrz\\Downloads\\dev_proj2_data")


Dataset 'annthyroid' has 138 duplicate rows (out of 7200. Removing them...
Dataset 'breastw' has 234 duplicate rows (out of 683. Removing them...
Dataset 'cardio' has 9 duplicate rows (out of 1831. Removing them...
Dataset 'glass' has 1 duplicate rows (out of 214. Removing them...
Dataset 'letter' has 2 duplicate rows (out of 1600. Removing them...
Dataset 'mammography' has 3335 duplicate rows (out of 11183. Removing them...
Dataset 'satimage-2' has 2 duplicate rows (out of 5803. Removing them...
Dataset 'thyroid' has 116 duplicate rows (out of 3772. Removing them...
Dataset 'vowels' has 4 duplicate rows (out of 1456. Removing them...
dict_keys(['annthyroid', 'arrhythmia', 'breastw', 'cardio', 'glass', 'letter', 'lympho', 'mammography', 'musk', 'pendigits', 'satellite', 'satimage-2', 'shuttle', 'speech', 'thyroid', 'vertebral', 'vowels', 'wine'])


# Helper functions

In [33]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score


def get_unified_outlier_scores(model, X, method_name):
    if method_name == "isolation-forest":
        return -model.score_samples(X)

    elif method_name == "one-class-svm":
        return -model.score_samples(X)

    elif method_name == "lof":
        return -model.negative_outlier_factor_

    elif method_name == "dbscan":
        labels = model.labels_
        return np.where(labels == -1, 1.0, 0.0)
    else:
        raise ValueError(f"Unknown method name: {method_name}")

def threshold_by_top_k(scores, contamination_rate):
    if contamination_rate <= 0 or contamination_rate >= 1:
        raise ValueError("Contamination rate must be between 0 and 1 exclusively.")

    n_samples = len(scores)
    k = int(np.ceil(contamination_rate * n_samples))
    threshold = np.partition(scores, -k)[-k]
    binary_preds = np.where(scores >= threshold, 1, 0)
    return binary_preds


def compute_evaluation_metrics(y_true, y_scores, y_pred=None, contamination=None):
    if y_pred is None:
        if contamination is None:
            raise ValueError("Must provide either 'y_pred' or a 'contamination' rate to calculate discrete metrics.")
        y_pred = threshold_by_top_k(y_scores, contamination)
    metrics = {
        "AUC-ROC": roc_auc_score(y_true, y_scores) if len(np.unique(y_true)) > 1 else np.nan,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0)
    }

    return metrics, y_pred

In [31]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
def parse_method_default(method_name):
    if method_name == "isolation-forest":
        return IsolationForest()
    elif method_name == "one-class-svm":
        return OneClassSVM()
    elif method_name == "dbscan":
        return DBSCAN()
    elif method_name == "lof":
        return LocalOutlierFactor()
    else:
        raise ValueError(f"Unknown method name: {method_name}")


# Performance evaluation at default hyperparams

In [40]:
default_results={
    method:{
        dataset:{}
        for dataset in all_datasets.keys()
    }
    for method in methods
}

In [41]:
for method in methods:
    for dataset in all_datasets.keys():
        model=parse_method_default(method)
        model.fit(all_datasets[dataset]["X"])
        scores=get_unified_outlier_scores(model, all_datasets[dataset]["X"], method)
        if method !='dbscan':
            preds=threshold_by_top_k(scores,np.mean(all_datasets[dataset]["y"])) #as y is binary vector of outlier indicators, its mean is the outlier ratio
        else:
            preds=scores
        default_results[method][dataset]=compute_evaluation_metrics(all_datasets[dataset]["y"],scores,y_pred=preds)[0]
        del model
    print(f"method {method} finished")

method one-class-svm finished
method isolation-forest finished
method dbscan finished
method lof finished


# Hyperparameter influence

In [46]:

param_grids = {
    "one-class-svm": {
        "nu": [0.01, 0.05, 0.1, 0.2],
        "kernel": ["rbf", "linear"],
        "gamma": ["scale", "auto"],
        "shrinking": [True, False]
    },
    "isolation-forest": {
        "n_estimators": [50, 100, 200],
        "max_features": [0.5, 1.0]
    },
    "lof": {
        "n_neighbors": [5, 15, 20, 50],
        "metric": ["euclidean", "manhattan"]
    }
}

hyperparam_results = {
    method: {dataset: [] for dataset in all_datasets.keys()}
    for method in ["one-class-svm", "isolation-forest", "lof"]
}

for dataset_name, data in all_datasets.items():
    if dataset_name !='wine':
        continue
    X = data["X"]
    y = data["y"]
    # Mean of binary outlier vector yields the contamination ratio
    contamination_rate = float(np.mean(y))

    grid_svm = param_grids["one-class-svm"]
    keys, values = zip(*grid_svm.items())
    for v in itertools.product(*values):
        params = dict(zip(keys, v))
        model = OneClassSVM(**params)
        model.fit(X)
        scores = get_unified_outlier_scores(model, X, "one-class-svm")
        preds = threshold_by_top_k(scores, contamination_rate)
        metrics, _ = compute_evaluation_metrics(y, scores, y_pred=preds)

        hyperparam_results["one-class-svm"][dataset_name].append({**params, **metrics})

    grid_if = param_grids["isolation-forest"]
    keys, values = zip(*grid_if.items())
    for v in itertools.product(*values):
        params = dict(zip(keys, v))
        model = IsolationForest(**params, random_state=42)
        model.fit(X)
        scores = get_unified_outlier_scores(model, X, "isolation-forest")
        preds = threshold_by_top_k(scores, contamination_rate)
        metrics, _ = compute_evaluation_metrics(y, scores, y_pred=preds)

        hyperparam_results["isolation-forest"][dataset_name].append({**params, **metrics})

    grid_lof = param_grids["lof"]
    keys, values = zip(*grid_lof.items())
    for v in itertools.product(*values):
        params = dict(zip(keys, v))
        if params["n_neighbors"] < len(X):
            model = LocalOutlierFactor(n_neighbors=params["n_neighbors"], metric=params["metric"])
            model.fit(X)
            scores = get_unified_outlier_scores(model, X, "lof")
            preds = threshold_by_top_k(scores, contamination_rate)
            metrics, _ = compute_evaluation_metrics(y, scores, y_pred=preds)

            hyperparam_results["lof"][dataset_name].append({**params, **metrics})

print("Continuous-score methods tuning completed successfully.")

Continuous-score methods tuning completed successfully.
